# 02 — Self-Attention Mathematics (Week 2)

Step-by-step walkthrough of scaled dot-product self-attention on an mRNA
sequence, matching `src/attention.py`:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import numpy as np

from attention import (EMBED_DIM, encode_sequence, multi_head_attention,
                       plot_attention, self_attention)

sequence = 'ACGUACGUACGGCUA'
X = encode_sequence(sequence)
print(f'Sequence length: {len(sequence)}, embedding dim: {EMBED_DIM}')
print(f'X shape: {X.shape}  (each row = one base embedded as an {EMBED_DIM}-D vector)')

## Step 1 — Q, K, V projections

Each position's embedding is linearly projected three ways:
- **Q** — what the position is *looking for*
- **K** — what the position *offers* to be matched
- **V** — the *content* contributed once matched

In [ ]:
rng = np.random.default_rng(42)
W_q = rng.standard_normal((EMBED_DIM, EMBED_DIM)) * 0.1
W_k = rng.standard_normal((EMBED_DIM, EMBED_DIM)) * 0.1
W_v = rng.standard_normal((EMBED_DIM, EMBED_DIM)) * 0.1

Q, K, V = X @ W_q, X @ W_k, X @ W_v
print('Q, K, V shapes:', Q.shape, K.shape, V.shape)

## Step 2 — scaled similarity scores and softmax

`Q · Kᵀ` scores every position against every other position. Dividing by `√dₖ`
keeps the logits small enough that softmax gradients don't vanish. Softmax turns
each row into a probability distribution that sums to 1.

In [ ]:
scores = (Q @ K.T) / np.sqrt(EMBED_DIM)
print(f'Raw score range: [{scores.min():.3f}, {scores.max():.3f}]')
print(f'After /sqrt(d_k): the spread stays small -> softmax stays responsive')

weights, output = self_attention(X)
assert np.allclose(weights.sum(axis=1), 1.0), 'rows must sum to 1'
print('Row sums (must all be 1.0):', np.round(weights.sum(axis=1)[:5], 6))
print('Attention output shape:', output.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(weights, cmap='viridis')
ax.set_xticks(range(len(sequence))); ax.set_yticks(range(len(sequence)))
ax.set_xticklabels(list(sequence)); ax.set_yticklabels(list(sequence))
ax.set_xlabel('Attending TO position'); ax.set_ylabel('Attending FROM position')
ax.set_title('Self-attention weights — block pattern from the repeating ACGU motif')
fig.colorbar(im, label='Attention weight')
fig.tight_layout(); plt.show()

print('Identical bases share embeddings, so positions 3 apart (the ACGU repeat)')
print('attend strongly to each other — a visible block/checkerboard pattern.')

## Step 4 — Bonus: multi-head attention

Run the mechanism with 4 independent Q/K/V weight sets and concatenate the outputs.

In [ ]:
mh_weights, mh_output = multi_head_attention(X, num_heads=4)
print(f'{len(mh_weights)} heads, each {mh_weights[0].shape}; concatenated output {mh_output.shape}')

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for i, (ax, w) in enumerate(zip(axes, mh_weights), 1):
    ax.imshow(w, cmap='viridis')
    ax.set_title(f'Head {i}')
    ax.set_xticks(range(len(sequence))); ax.set_yticks(range(len(sequence)))
    ax.set_xticklabels(list(sequence), fontsize=6); ax.set_yticklabels(list(sequence), fontsize=6)
fig.suptitle('Multi-head attention — each head sees a different random projection')
fig.tight_layout(); plt.show()